# 강의 08 - 다중 에이전트 디자인 패턴


## 설정


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 다중 에이전트 시스템이 필요한 이유

여행 계획과 같은 실제 작업에는 물류, 지역 지식, 예산 등 다양한 전문 지식이 필요합니다. 모든 것을 혼자 처리하려는 단일 에이전트는 빠르게 다루기 어려워집니다.

다중 에이전트 시스템은 **전문화**를 통해 이를 해결합니다: 각 에이전트는 한 분야에 집중하여 일반 주의자보다 더 높은 품질의 결과를 만들어냅니다. 또한 **확장성**도 향상시킵니다 — 새 에이전트(예: 항공 전문가, 레스토랑 평론가)를 기존 워크플로를 다시 작성하지 않고도 추가할 수 있습니다. 에이전트들은 구조화된 파이프라인을 통해 구성되어, 문맥을 한 에이전트에서 다음 에이전트로 전달합니다.


## 전문 에이전트 만들기


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## 순차 워크플로우 구축하기

`WorkflowBuilder`를 사용하면 에이전트를 방향성 그래프로 연결할 수 있습니다. 여기서는 간단한 두 단계 파이프라인을 만듭니다: **TravelPlanner**가 일정을 초안 작성하고, 그다음 **TravelConcierge**가 이를 검토하고 향상시킵니다.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 워크플로우에 에이전트 추가하기

멀티 에이전트 패턴의 가장 큰 장점 중 하나는 확장하기 매우 쉽다는 점입니다. 아래에서는 여행자의 예산과 계획을 비교하고, 비용 초과 가능성이 있는 항목에 표시를 하며, 비용 절감 대안을 제안하는 **BudgetReviewer** 에이전트를 추가합니다. 이제 워크플로우는 세 개의 에이전트를 순차적으로 실행합니다:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## 요약

이 수업에서는 다음을 배웠습니다:

1. **전문화된 에이전트 생성하기** — 각기 집중된 역할(기획, 컨시어지, 예산 검토)을 가진 에이전트.
2. **`WorkflowBuilder`와 `add_edge`를 사용해 에이전트들을 순차적 워크플로에 연결하기**.
3. **다중 에이전트 파이프라인에서 출력 스트리밍하기**, 누가 말하는지 추적.
4. **기존 에이전트를 수정하지 않고 새로운 에이전트를 체인에 추가해 워크플로 확장하기**.

다중 에이전트 디자인 패턴은 각 에이전트를 단순하게 유지하면서 단일 에이전트가 달성할 수 있는 것보다 더 풍부하고 철저히 검토된 결과를 만듭니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:  
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 위해 노력하고 있으나, 자동 번역에는 오류나 부정확한 내용이 포함될 수 있음을 알려드립니다. 원문은 해당 언어의 원본 문서를 권위 있는 출처로 간주해야 합니다. 중요한 정보의 경우 전문 번역가에 의한 번역을 권장합니다. 본 번역물의 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
